In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

#  Merged dataset location
Merged_DIR = Path(
    r"C:\Users\15195\Desktop\coding part\dataset merged"
)

# EDA output location
OUTPUT_DIR = Path(
    r"C:\Users\15195\Desktop"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Read the transaction-product merged dataset
transaction_product_path = Merged_DIR / "transaction_product_merged.csv"

transaction_product = pd.read_csv(transaction_product_path)

print("Transaction-product dataset shape:", transaction_product.shape)
print("\nColumns:")
print(transaction_product.columns.tolist())

transaction_product.head()

c:\Users\15195\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Transaction-product dataset shape: (2595732, 18)

Columns:
['household_key', 'BASKET_ID', 'DAY', 'PRODUCT_ID', 'QUANTITY', 'SALES_VALUE', 'STORE_ID', 'RETAIL_DISC', 'TRANS_TIME', 'WEEK_NO', 'COUPON_DISC', 'COUPON_MATCH_DISC', 'MANUFACTURER', 'DEPARTMENT', 'BRAND', 'COMMODITY_DESC', 'SUB_COMMODITY_DESC', 'CURR_SIZE_OF_PRODUCT']


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC,MANUFACTURER,DEPARTMENT,BRAND,COMMODITY_DESC,SUB_COMMODITY_DESC,CURR_SIZE_OF_PRODUCT
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0,69,PRODUCE,Private,POTATOES,POTATOES RUSSET (BULK&BAG),5 LB
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0,2,PRODUCE,National,ONIONS,ONIONS SWEET (BULK&BAG),40 LB
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0,69,PRODUCE,Private,VEGETABLES - ALL OTHERS,CELERY,
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0,2,PRODUCE,National,TROPICAL FRUIT,BANANAS,40 LB
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0,69,PRODUCE,Private,ORGANICS FRUIT & VEGETABLES,ORGANIC CARROTS,1 LB


In [2]:
# Data quality checks before EDA
print("Missing values in EDA-relevant variables:")
print(
    transaction_product[
        [
            "household_key",
            "BASKET_ID",
            "DAY",
            "PRODUCT_ID",
            "QUANTITY",
            "SALES_VALUE",
            "RETAIL_DISC",
            "COUPON_DISC",
            "COUPON_MATCH_DISC",
            "DEPARTMENT",
            "COMMODITY_DESC"
        ]
    ].isnull().sum()
)

print("\nExact duplicate rows:")
print(transaction_product.duplicated().sum())

print("\nNumber of unique households:",
      transaction_product["household_key"].nunique())

print("Number of unique baskets:",
      transaction_product["BASKET_ID"].nunique())

print("Number of unique products:",
      transaction_product["PRODUCT_ID"].nunique())

Missing values in EDA-relevant variables:
household_key        0
BASKET_ID            0
DAY                  0
PRODUCT_ID           0
QUANTITY             0
SALES_VALUE          0
RETAIL_DISC          0
COUPON_DISC          0
COUPON_MATCH_DISC    0
DEPARTMENT           0
COMMODITY_DESC       0
dtype: int64

Exact duplicate rows:
0

Number of unique households: 2500
Number of unique baskets: 276484
Number of unique products: 92339


In [3]:
# Basic EDA indicators

# 1. Total sales
total_sales = transaction_product["SALES_VALUE"].sum()

# 2. Number of households
number_of_households = transaction_product["household_key"].nunique()

# 3. Number of baskets
number_of_baskets = transaction_product["BASKET_ID"].nunique()

# 4. Average basket value
basket_sales = (
    transaction_product
    .groupby("BASKET_ID", as_index=False)
    .agg(basket_sales=("SALES_VALUE", "sum"))
)

average_basket_value = basket_sales["basket_sales"].mean()

# 5. Average basket size based on total quantity
basket_size_quantity = (
    transaction_product
    .groupby("BASKET_ID", as_index=False)
    .agg(basket_quantity=("QUANTITY", "sum"))
)

average_basket_size_quantity = (
    basket_size_quantity["basket_quantity"].mean()
)

# 6. Average basket size based on unique commodity categories
basket_size_category = (
    transaction_product
    .groupby("BASKET_ID", as_index=False)
    .agg(
        basket_category_count=(
            "COMMODITY_DESC",
            "nunique"
        )
    )
)

average_basket_size_category = (
    basket_size_category["basket_category_count"].mean()
)

# 7. Total discount:
# retail loyalty discount + manufacturer coupon discount
# + retailer coupon-match discount
total_discount = (
    transaction_product["RETAIL_DISC"].abs().sum()
    + transaction_product["COUPON_DISC"].abs().sum()
    + transaction_product["COUPON_MATCH_DISC"].abs().sum()
)

# 8. Overall discount ratio
discount_ratio = total_discount / total_sales

# 9. Create EDA summary table
eda_summary = pd.DataFrame({
    "indicator": [
        "total_sales",
        "number_of_households",
        "number_of_baskets",
        "average_basket_value",
        "average_basket_size_quantity",
        "average_basket_size_category",
        "total_discount",
        "discount_ratio"
    ],
    "value": [
        total_sales,
        number_of_households,
        number_of_baskets,
        average_basket_value,
        average_basket_size_quantity,
        average_basket_size_category,
        total_discount,
        discount_ratio
    ]
})

eda_summary

,indicator,value
0,total_sales,8.057463e+06
1,number_of_households,2.500000e+03
2,number_of_baskets,2.764840e+05
3,average_basket_value,2.914260e+01
4,average_basket_size_quantity,9.428597e+02
5,average_basket_size_category,6.930235e+00
6,total_discount,1.448539e+06
7,discount_ratio,1.797761e-01


In [4]:
# Save eda_summary.csv
eda_summary.to_csv(
    OUTPUT_DIR / "eda_summary.csv",
    index=False
)

print(" ^ > < ^ ! Saved:", OUTPUT_DIR / "eda_summary.csv")

 ^ > < ^ ! Saved: C:\Users\15195\Desktop\eda_summary.csv


In [5]:
# Department-level summary
department_summary = (
    transaction_product
    .groupby("DEPARTMENT", as_index=False)
    .agg(
        total_sales=("SALES_VALUE", "sum"),
        total_quantity=("QUANTITY", "sum"),
        basket_count=("BASKET_ID", "nunique"),
        household_count=("household_key", "nunique"),
        product_count=("PRODUCT_ID", "nunique"),
        category_count=("COMMODITY_DESC", "nunique")
    )
)

# Department sales share
department_summary["department_sales_share"] = (
    department_summary["total_sales"]
    / department_summary["total_sales"].sum()
)

# Household penetration:
# proportion of all households that purchased from the department
total_households = transaction_product["household_key"].nunique()

department_summary["department_penetration"] = (
    department_summary["household_count"]
    / total_households
)

# Average department sales per basket containing that department
department_summary["average_sales_per_basket"] = (
    department_summary["total_sales"]
    / department_summary["basket_count"]
)

department_summary = department_summary.sort_values(
    by="total_sales",
    ascending=False
).reset_index(drop=True)

department_summary.head(20)

,DEPARTMENT,total_sales,total_quantity,basket_count,household_count,product_count,category_count,department_sales_share,department_penetration,average_sales_per_basket
0,GROCERY,4093814.14,2194762,215411,2500,39019,94,0.508077,1.0000,19.004666
1,DRUG GM,1055358.03,353844,118195,2491,31523,91,0.130979,0.9964,8.928957
2,PRODUCE,557452.11,319993,89026,2456,3117,32,0.069185,0.9824,6.261678
3,MEAT,548786.81,119113,53401,2351,2544,15,0.068109,0.9404,10.276714
4,KIOSK-GAS,544222.28,221254887,22056,1371,16,2,0.067543,0.5484,24.674568
5,MEAT-PCKGD,412436.77,148148,58466,2392,2427,10,0.051187,0.9568,7.054301
6,DELI,260866.51,67026,35639,2248,2354,13,0.032376,0.8992,7.319692
7,PASTRY,121739.86,49820,30099,2281,2149,9,0.015109,0.9124,4.044648
8,MISC SALES TRAN,119960.04,36080860,5992,1393,88,1,0.014888,0.5572,20.020033
9,NUTRITION,97669.04,43253,16820,1719,2913,25,0.012122,0.6876,5.806721


In [6]:
# Save department summary
department_summary.to_csv(
    OUTPUT_DIR / "department_summary.csv",
    index=False
)

print(" * V * v Saved:", OUTPUT_DIR / "department_summary.csv")

 * V * v Saved: C:\Users\15195\Desktop\department_summary.csv


In [7]:
# Commodity-category-level summary
category_summary = (
    transaction_product
    .groupby("COMMODITY_DESC", as_index=False)
    .agg(
        total_sales=("SALES_VALUE", "sum"),
        total_quantity=("QUANTITY", "sum"),
        basket_count=("BASKET_ID", "nunique"),
        household_count=("household_key", "nunique"),
        product_count=("PRODUCT_ID", "nunique"),
        department_count=("DEPARTMENT", "nunique")
    )
)

# Category sales share
category_summary["category_sales_share"] = (
    category_summary["total_sales"]
    / category_summary["total_sales"].sum()
)

# Household penetration
category_summary["category_penetration"] = (
    category_summary["household_count"]
    / total_households
)

# Average sales per basket containing that category
category_summary["average_sales_per_basket"] = (
    category_summary["total_sales"]
    / category_summary["basket_count"]
)

category_summary = category_summary.sort_values(
    by="total_sales",
    ascending=False
).reset_index(drop=True)

category_summary.head(20)

,COMMODITY_DESC,total_sales,total_quantity,basket_count,household_count,product_count,department_count,category_sales_share,category_penetration,average_sales_per_basket
0,COUPON/MISC ITEMS,639878.56,257218037,27705,1985,128,9,0.079414,0.7940,23.096140
1,SOFT DRINKS,327647.30,160637,71699,2405,1704,1,0.040664,0.9620,4.569761
2,BEEF,312103.22,65576,36733,2238,1109,1,0.038735,0.8952,8.496535
3,FLUID MILK PRODUCTS,205356.05,116192,69278,2421,455,1,0.025486,0.9684,2.964232
4,CHEESE,189528.18,96402,46898,2350,1096,1,0.023522,0.9400,4.041285
5,FRZN MEAT/MEAT DINNERS,160517.17,79391,22777,2069,1268,1,0.019922,0.8276,7.047336
6,BAG SNACKS,148375.16,80824,42037,2348,1523,1,0.018415,0.9392,3.529632
7,BEERS/ALES,147344.45,19863,15527,1606,795,2,0.018287,0.6424,9.489563
8,FROZEN PIZZA,146037.25,67225,23174,2052,901,1,0.018124,0.8208,6.301771
9,BAKED BREAD/BUNS/ROLLS,145930.85,99713,60311,2400,1169,1,0.018111,0.9600,2.419639


In [8]:
# Save Category summary
category_summary.to_csv(
    OUTPUT_DIR / "category_summary.csv",
    index=False
)

print(" (@ V @ )a Saved:", OUTPUT_DIR / "category_summary.csv")

 (@ V @ )a Saved: C:\Users\15195\Desktop\category_summary.csv


In [9]:
# Top 11 departments and commodity categories by sales
top11_departments = department_summary.head(11)
top11_categories = category_summary.head(11)

print("Top 11 Departments by Sales")
display(top11_departments)

print("Top 11 Commodity Categories by Sales")
display(top11_categories)

top11_departments.to_csv(
    OUTPUT_DIR / "top11_departments.csv",
    index=False
)

top11_categories.to_csv(
    OUTPUT_DIR / "top11_categories.csv",
    index=False
)

print("^ >< ^ Saved:", OUTPUT_DIR / "top11_departments.csv")
print(" U * v * U Saved:", OUTPUT_DIR / "top11_categories.csv")

Top 11 Departments by Sales


,DEPARTMENT,total_sales,total_quantity,basket_count,household_count,product_count,category_count,department_sales_share,department_penetration,average_sales_per_basket
0,GROCERY,4093814.14,2194762,215411,2500,39019,94,0.508077,1.0000,19.004666
1,DRUG GM,1055358.03,353844,118195,2491,31523,91,0.130979,0.9964,8.928957
2,PRODUCE,557452.11,319993,89026,2456,3117,32,0.069185,0.9824,6.261678
3,MEAT,548786.81,119113,53401,2351,2544,15,0.068109,0.9404,10.276714
4,KIOSK-GAS,544222.28,221254887,22056,1371,16,2,0.067543,0.5484,24.674568
5,MEAT-PCKGD,412436.77,148148,58466,2392,2427,10,0.051187,0.9568,7.054301
6,DELI,260866.51,67026,35639,2248,2354,13,0.032376,0.8992,7.319692
7,PASTRY,121739.86,49820,30099,2281,2149,9,0.015109,0.9124,4.044648
8,MISC SALES TRAN,119960.04,36080860,5992,1393,88,1,0.014888,0.5572,20.020033
9,NUTRITION,97669.04,43253,16820,1719,2913,25,0.012122,0.6876,5.806721


Top 11 Commodity Categories by Sales


,COMMODITY_DESC,total_sales,total_quantity,basket_count,household_count,product_count,department_count,category_sales_share,category_penetration,average_sales_per_basket
0,COUPON/MISC ITEMS,639878.56,257218037,27705,1985,128,9,0.079414,0.7940,23.096140
1,SOFT DRINKS,327647.30,160637,71699,2405,1704,1,0.040664,0.9620,4.569761
2,BEEF,312103.22,65576,36733,2238,1109,1,0.038735,0.8952,8.496535
3,FLUID MILK PRODUCTS,205356.05,116192,69278,2421,455,1,0.025486,0.9684,2.964232
4,CHEESE,189528.18,96402,46898,2350,1096,1,0.023522,0.9400,4.041285
5,FRZN MEAT/MEAT DINNERS,160517.17,79391,22777,2069,1268,1,0.019922,0.8276,7.047336
6,BAG SNACKS,148375.16,80824,42037,2348,1523,1,0.018415,0.9392,3.529632
7,BEERS/ALES,147344.45,19863,15527,1606,795,2,0.018287,0.6424,9.489563
8,FROZEN PIZZA,146037.25,67225,23174,2052,901,1,0.018124,0.8208,6.301771
9,BAKED BREAD/BUNS/ROLLS,145930.85,99713,60311,2400,1169,1,0.018111,0.9600,2.419639


^ >< ^ Saved: C:\Users\15195\Desktop\top11_departments.csv
 U * v * U Saved: C:\Users\15195\Desktop\top11_categories.csv
